In [1]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

assert torch.backends.mps.is_available()
device = torch.device("mps")

/opt/homebrew/Caskroom/miniconda/base/envs/fieldbeat/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ALIAS_FILTER_TEMPLATE = f"""
Tu tarea consiste en seleccionar, para cada "pregunta", los alias que representen correctamente el **nombre, denominación real o sobrenombre reconocido** de la respuesta correcta.  

Debes eliminar alias que sean descripciones, reformulaciones de la pregunta, o frases tautológicas (por ejemplo, "Capital de Chile" cuando la pregunta es "¿Cuál es la capital de Chile?").  

Quédate solo con los **nombres válidos del objeto**, incluyendo:
- Variantes ortográficas o lingüísticas ("Nueva York", "New York").
- Nombres históricos o alternativos ("Ciudad de los Reyes" para Lima).
- **Sobrenombres, apodos o títulos poéticos reconocidos** de la entidad ("La Ciudad Blanca", "La Perla del Caribe").
Pero **no** conserves explicaciones, reformulaciones o descripciones genéricas.

---

### MODO DE RAZONAMIENTO (ejemplo a seguir)

1. Analiza la **intención de la pregunta** (qué tipo de entidad busca: persona, lugar, objeto, organización, etc.).
2. Considera la **respuesta correcta**, que te indica exactamente qué entidad debe representar el alias.
3. Examina cada alias:
   - Conserva los que sean **nombres propios, denominaciones oficiales o sobrenombres reconocidos** de la respuesta correcta.
   - Elimina los que **repiten palabras clave de la pregunta** (por ejemplo: “capital”, “país”, “moneda”, “presidente”, “río”).
   - Elimina los que son **descripciones, definiciones o frases contextuales**.
4. Si varios alias son válidos, conserva todos los que correspondan a nombres reales o variantes legítimas.
5. Devuelve solo los alias válidos en formato de lista (por ejemplo: ["Lima", "Ciudad de los Reyes"]). Devuelve una lista vacía si ninguno es válido (por ejemplo: []).

---

### EJEMPLOS CON RAZONAMIENTO

**Ejemplo 1**
pregunta: ¿Cuál es la capital de Chile?  
respuesta correcta: Santiago  
posibles aliases: Gran Santiago, Santiago de Chile, Capital de Chile, La capital chilena  
razonamiento:  
“Gran Santiago” y “Santiago de Chile” son nombres propios válidos de la ciudad.  
“Capital de Chile” y “La capital chilena” son descripciones que repiten la pregunta.  
output: ["Gran Santiago", "Santiago de Chile"]

---

**Ejemplo 2**
pregunta: ¿Cuál es la capital de Perú?  
respuesta correcta: Lima  
posibles aliases: Lima, Ciudad de los Reyes, Capital del Perú  
razonamiento:  
“Lima” es el nombre oficial de la ciudad,  
“Ciudad de los Reyes” es su nombre histórico y también un alias válido.  
“Capital del Perú” repite la pregunta y debe eliminarse.  
output: ["Lima", "Ciudad de los Reyes"]

---

**Ejemplo 3**
pregunta: ¿Qué país tiene como capital a Tokio?  
respuesta correcta: Japón  
posibles aliases: País del Sol Naciente, nación japonesa, país asiático  
razonamiento:  
“País del Sol Naciente” y “nación japonesa” son denominaciones válidas del país.  
“país asiático” es una descripción genérica.  
output: ["Japón", "País del Sol Naciente"]

---

**Ejemplo 4**
pregunta: ¿Qué ciudad es conocida como “La Ciudad Blanca”?  
respuesta correcta: Arequipa  
posibles aliases: Arequipa, La Ciudad Blanca, Ciudad de Arequipa, urbe arequipeña  
razonamiento:  
“Arequipa” y “La Ciudad Blanca” son nombres o sobrenombres válidos de la misma ciudad.  
“Ciudad de Arequipa” repite la pregunta, y “urbe arequipeña” es una descripción genérica.  
output: ["Arequipa", "La Ciudad Blanca"]

---

### NUEVO CASO

Ahora aplica el mismo razonamiento anterior, pero **sin mostrar el razonamiento**, solo entrega el resultado final como una lista.

pregunta: {{pregunta}}  
respuesta correcta: {{respuesta_correcta}}  
posibles aliases: {{aliases}}  

output:
""".strip()



In [3]:
import time
import json

MISTRAL_API_URL = "http://localhost:1234/v1/chat/completions"  # LM Studio API local

In [4]:
import json, re

def _safe_json_list(text: str) -> list[str]:
    """Parsea una lista JSON de forma robusta desde texto ruidoso del LLM."""
    if not isinstance(text, str):
        return []

    t = text.strip()

    # 1) quita code fences y prefijos tipo 'output:'
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r'^\s*(output|salida)\s*:\s*', '', t, flags=re.IGNORECASE)

    # 2) si no hay '['...']', intenta extraer el primer bloque entre corchetes
    if not t.lstrip().startswith('['):
        m = re.search(r'\[[\s\S]*?\]', t)
        if m:
            t = m.group(0)

    # 3) normaliza comillas tipográficas y simples
    t = t.replace("“", '"').replace("”", '"')
    # si es una lista y solo hay comillas simples, cámbialas por dobles
    if t.startswith('[') and "'" in t:
        # pero evita romper JSON válido que ya tenga dobles
        if '"' not in t or t.count("'") > t.count('"'):
            t = t.replace("'", '"')

    # 4) intenta json.loads directamente
    try:
        data = json.loads(t)
        if isinstance(data, list):
            out = [s.strip() for s in data if isinstance(s, str) and s.strip()]
            # dedup en orden
            seen, dedup = set(), []
            for s in out:
                if s not in seen:
                    seen.add(s); dedup.append(s)
            return dedup
    except Exception:
        pass

    # 5) fallback: extrae strings entre comillas dobles
    items = re.findall(r'"([^"]+)"', t)
    if items:
        out = [s.strip() for s in items if s.strip()]
        seen, dedup = set(), []
        for s in out:
            if s not in seen:
                seen.add(s); dedup.append(s)
        return dedup

    # 6) último recurso: split por coma dentro del primer bloque [...]
    m = re.search(r'\[([\s\S]*?)\]', t)
    core = m.group(1) if m else t
    parts = [p.strip(" \t\n\r\"'") for p in core.split(",")]
    out = [p for p in parts if p]
    seen, dedup = set(), []
    for s in out:
        if s not in seen:
            seen.add(s); dedup.append(s)
    return dedup

In [5]:
import requests

def generate_aliases(pregunta, respuesta_correcta, respuesta_entregada):

    prompt = ALIAS_FILTER_TEMPLATE.format(
		pregunta=pregunta,
		respuesta_correcta=respuesta_correcta,
		aliases=respuesta_entregada
	)

    payload = {
        "model": "mistral",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 4000,
        "stream": False
    }

    response = requests.post(MISTRAL_API_URL, json=payload, stream=False)

    respuesta = response.json()
    #return list of strings
    return _safe_json_list(respuesta['choices'][0]['message']['content'])

In [7]:
filtered1 = generate_aliases("Capital de Ecuador", "Quito", ["Quito", "Capital de Ecuador"])
filtered2 = generate_aliases("Capital de Argentina", "Buenos Aires", ["Buenos Aires", "Ciudad Autónoma de Buenos Aires", "Capital de Argentina", "La Reina del Plata"])
filtered3 = generate_aliases("Cuál es la figura Argentina clave en la revolución cubana", "Ernesto 'Che' Guevara", ["Ernesto Guevara", "Che Guevara", "El Che", "Figura Argentina en la revolución cubana"])
print(filtered1)
print(filtered2)
print(filtered3)

['Quito']
['Buenos Aires', 'Ciudad Autónoma de Buenos Aires', 'La Reina del Plata']
['Ernesto Guevara', 'Che Guevara', 'El Che']


In [8]:
import json
import ast

def parse_list(x):
    """Convierte strings tipo '["a","b"]' en listas reales"""
    if isinstance(x, str) and x.startswith("["):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return x if isinstance(x, list) else []

In [9]:
INPUT_PATH = "QA_dataset_aliases_parcial.csv"
OUTPUT_PATH = "QA_dataset_aliases_filtrados.csv"
TMP_PATH = OUTPUT_PATH + ".tmp"
PROGRESS_PATH = "progress.idx"
SAVE_EVERY = 100  # filas

In [10]:
print("Leyendo dataset...")
if os.path.exists(OUTPUT_PATH):
    print(f"Reanudando desde archivo existente: {OUTPUT_PATH}")
    df = pd.read_csv(OUTPUT_PATH)
else:
    df = pd.read_csv(INPUT_PATH)
df["respuestas"] = df["respuestas"].apply(parse_list)
df["respuestas_aliases"] = df["respuestas_aliases"].apply(parse_list)
print(f"Filas totales: {len(df)}\n")

Leyendo dataset...
Reanudando desde archivo existente: QA_dataset_aliases_filtrados.csv
Filas totales: 56657



In [11]:
def normalize_aliases(x):
    # NaN / None -> []
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    # Si no es lista, envuélvelo (lo dejamos como lista de 1 y luego lo afinamos)
    if not isinstance(x, list):
        return [x]
    return x

def has_any_alias(container):
    """
    True si hay al menos un string no vacío
    en container (que puede ser lista de listas o lista simple).
    """
    if not isinstance(container, list):
        return isinstance(container, str) and container.strip()
    for x in container:
        if isinstance(x, list):
            if any(isinstance(s, str) and s.strip() for s in x):
                return True
        elif isinstance(x, str) and x.strip():
            return True
    return False

In [12]:
# índice desde el cual reanudar
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r") as f:
        last_idx = int(f.read().strip() or 0)
else:
    last_idx = 0

In [13]:
print(f"Filas totales: {len(df)} | Reanudando desde idx = {last_idx}\n")

Filas totales: 56657 | Reanudando desde idx = 33306



In [16]:
processed_since_save = 0

# ==== procesamiento ====
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Procesando filas"):
    if idx < last_idx:
        continue  # ya hecho en un run anterior

    pregunta = row["pregunta"]
    respuestas = row["respuestas"]
    respuestas_aliases = normalize_aliases(row["respuestas_aliases"])

    # si no hay aliases en absoluto, deja como está
    if not has_any_alias(respuestas_aliases):
        df.at[idx, "respuestas_aliases"] = respuestas_aliases
    else:
        # asegurar respuestas lista
        if not isinstance(respuestas, list):
            respuestas = [respuestas]

        # alinear longitudes
        if len(respuestas_aliases) < len(respuestas):
            respuestas_aliases += [[] for _ in range(len(respuestas) - len(respuestas_aliases))]
        elif len(respuestas_aliases) > len(respuestas):
            respuestas_aliases = respuestas_aliases[:len(respuestas)]

        pairs = list(zip(respuestas, respuestas_aliases))
        filtered_aliases_list = []

        for respuesta_correcta, alias_list in pairs:
            alias_list = [a for a in alias_list if isinstance(a, str) and a.strip()]
            if not alias_list or not respuesta_correcta:
                filtered_aliases_list.append([])
                continue

            try:
                filtered = generate_aliases(pregunta, respuesta_correcta, alias_list)
            except Exception as e:
                print(f" Error en fila {idx}: {e}")
                filtered = []
            filtered_aliases_list.append(filtered)

        df.at[idx, "respuestas_aliases"] = filtered_aliases_list

    processed_since_save += 1
    last_idx = idx + 1  # próximo índice desde el que continuar
    # guardar progreso de índice SIEMPRE (barato y seguro)
    with open(PROGRESS_PATH, "w") as f:
        f.write(str(last_idx))

    # guardado periódico del CSV
    if processed_since_save % SAVE_EVERY == 0:
        df.to_csv(TMP_PATH, index=False)
        os.replace(TMP_PATH, OUTPUT_PATH)
        # mantener progreso consistente
        with open(PROGRESS_PATH, "w") as f:
            f.write(str(last_idx))
        tqdm.write(f"Guardado parcial. Último idx: {last_idx}")

# guardado final
df.to_csv(TMP_PATH, index=False)
os.replace(TMP_PATH, OUTPUT_PATH)
with open(PROGRESS_PATH, "w") as f:
    f.write(str(last_idx))
print(f" Procesamiento completo. Guardado en {OUTPUT_PATH}. Último idx: {last_idx}")

Procesando filas:  60%|█████▉    | 33949/56657 [01:24<04:06, 92.01it/s]   

Guardado parcial. Último idx: 33961


Procesando filas:  60%|██████    | 34058/56657 [03:27<3:14:44,  1.93it/s]

Guardado parcial. Último idx: 34061


Procesando filas:  60%|██████    | 34161/56657 [05:16<6:27:30,  1.03s/it]

...

Guardado parcial. Último idx: 56461


Procesando filas: 100%|█████████▉| 56561/56657 [4:54:35<01:21,  1.18it/s]

Guardado parcial. Último idx: 56561


Procesando filas: 100%|██████████| 56657/56657 [4:56:13<00:00,  3.19it/s]


 Procesamiento completo. Guardado en QA_dataset_aliases_filtrados.csv. Último idx: 56657


In [25]:
import pandas as pd
import ast

def _parse_list_safe(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.strip().startswith('['):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    if pd.isna(x):
        return []
    # si viniera un escalar suelto
    return [x]

def _count_aliases(aliases):
    """Cuenta strings no vacíos en una estructura lista o lista de listas."""
    aliases = _parse_list_safe(aliases)
    total = 0
    for a in aliases:
        if isinstance(a, list):
            total += sum(1 for x in a if isinstance(x, str) and x.strip())
        elif isinstance(a, str) and a.strip():
            total += 1
    return total

def comparar_aliases(df_original, df_filtrado, col="respuestas_aliases"):
    """
    Compara la cantidad de aliases entre dos DataFrames (antes/después).
    Devuelve un DataFrame con: idx, originales, filtrados, reducidos, reduccion_%.
    """
    n = min(len(df_original), len(df_filtrado))
    rows = []
    for i in range(n):
        orig = df_original.at[i, col] if col in df_original.columns else []
        filt = df_filtrado.at[i, col] if col in df_filtrado.columns else []
        orig_count = _count_aliases(orig)
        filt_count = _count_aliases(filt)
        reducidos = orig_count - filt_count
        reduccion_pct = (reducidos / orig_count * 100) if orig_count > 0 else 0.0
        rows.append((i, orig_count, filt_count, reducidos, reduccion_pct))

    resumen = pd.DataFrame(rows, columns=["idx", "originales", "filtrados", "reducidos", "reduccion_%"])

    total_orig = int(resumen["originales"].sum())
    total_filt = int(resumen["filtrados"].sum())
    total_reduc = total_orig - total_filt
    pct_global = (total_reduc / total_orig * 100) if total_orig > 0 else 0.0

    print("RESUMEN GLOBAL:")
    print(f"   Aliases originales: {total_orig}")
    print(f"   Aliases filtrados:  {total_filt}")
    print(f"   Reducidos:          {total_reduc} ({pct_global:.1f}% menos)")

    return resumen


In [26]:
df_filtrado = pd.read_csv(OUTPUT_PATH)
df_original = pd.read_csv(INPUT_PATH)
resumen_comparacion = comparar_aliases(df_original, df_filtrado, col="respuestas_aliases")
print(resumen_comparacion.head(10))  # muestra las primeras 10 filas del

RESUMEN GLOBAL:
   Aliases originales: 203623
   Aliases filtrados:  130741
   Reducidos:          72882 (35.8% menos)
   idx  originales  filtrados  reducidos  reduccion_%
0    0           4          2          2    50.000000
1    1           1          1          0     0.000000
2    2           1          1          0     0.000000
3    3           0          0          0     0.000000
4    4           3          3          0     0.000000
5    5           0          0          0     0.000000
6    6           2          2          0     0.000000
7    7           7          3          4    57.142857
8    8           7          2          5    71.428571
9    9           0          0          0     0.000000
